# Milestone 2 · Session 3 of 3 — RQ-M fairness: MOMENT / TTM / Moirai-2

Finishes the C-MAPSS vertical slice (IMPLEMENTATION_PLAN §5) together with its two sibling notebooks. This session runs ONLY the RQ-M representation-fairness ablation for the three backbones the probe sessions don't cover — **one model per runtime cycle** (the stacks are mutually incompatible), three cycles total. Each cycle ≈ two new embedding passes (FD001 + FD004 common-representation caches); the native arm is a cache hit.

Session split (one backbone per runtime — `requirements/README.md`):

1. `timesfm_probes.ipynb` — **TimesFM 2.5**: RQ-A/C/E/H factor probes **with the shared baselines** + RQ-M fairness (TimesFM).
2. `chronos_probes_zeroshot.ipynb` — **Chronos-2**: the same probes models-only + RQ-M fairness (Chronos-2) + **RQ-Z zero-shot**.
3. `fairness_moment_ttm_moirai.ipynb` — RQ-M fairness for **MOMENT / TTM / Moirai-2** (one model per runtime cycle).

The probe roster (top-2 TSFMs TimesFM 2.5 + Chronos-2, foils gbm + minirocket, NN lstm) is the `probe_roster` resolution recorded in CHANGES.md §46.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU`; use a **fresh runtime** (the backbones must never share an environment); run top-to-bottom. Every stage is restartable — re-running resumes and skips completed cells; levels marked *(cache hit)* reuse the §45 campaign caches on Drive.


In [ ]:
# 1) clone the repo (shallow) — same pattern as notebooks/campaign/* Stage A
%cd /content
!git clone --depth 1 --branch main https://github.com/blozanod/Predictive-Maintenance-LSTM.git 2>/dev/null || echo "(already cloned — reusing)"
%cd /content/Predictive-Maintenance-LSTM
import sys; sys.path.insert(0, '/content/Predictive-Maintenance-LSTM')   # import src.* from the fresh clone


In [ ]:
# 2) pick THIS runtime cycle's model (uncomment ONE), then run the rest top-to-bottom.
#    Repeat the cycle per model: set MODEL -> Runtime ▸ Disconnect and delete runtime ->
#    run all again. Fairness for TimesFM/Chronos-2 lives in the session-1/2 notebooks.
MODEL = 'AutonLab/MOMENT-1-large'
# MODEL = 'ibm-granite/granite-timeseries-ttm-r2'
# MODEL = 'Salesforce/moirai-2.0-R-small'

REQ, FLAGS = {
    'AutonLab/MOMENT-1-large':               ('moment', '--no-deps'),
    'ibm-granite/granite-timeseries-ttm-r2': ('ttm', ''),
    'Salesforce/moirai-2.0-R-small':         ('moirai', ''),
}[MODEL]
TAG = REQ
print(MODEL, '→ requirements/' + REQ + '.txt', FLAGS)


In [ ]:
# 3) install ONLY this model's isolated stack. NOTE (ttm/moirai): the install CHANGES
#    torch/torchvision — after it finishes, Runtime ▸ Restart session, then re-run from
#    the top (clone + pip are cached, so the restart is quick) before importing src.*.
!pip install {FLAGS} -r requirements/{REQ}.txt


### No dependency top-up needed in this session

Unlike the two probe notebooks, this one needs **nothing beyond the backbone stack**:
`run_representation_fairness` trains its head with **MSE only** (hardcoded, `src/sweep.py`)
and runs **no baselines**, so neither `coral-pytorch` (CORN) nor `lightgbm` (gbm) is
imported. Keeping this runtime at exactly `requirements/<model>.txt` is what protects
Moirai-2's `torch==2.4.1` and TTM's `torch==2.10.0` pins.


In [ ]:
# 4) mount Google Drive — the SAME folder the §45 campaign wrote the caches/results to
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 5) the CANONICAL config — identical to the §45 campaign notebooks for every cache-key
#    field, so the native fairness arm is a CACHE HIT on the campaign caches.
from src.config import Config

DRIVE = '/content/drive/MyDrive/pdm_tsfm'   # SAME Drive folder as the §45 campaign
RESULTS = f'{DRIVE}/results'

config = Config(
    data_root='Data',
    cache_dir=f'{DRIVE}/cache',
    results_dir=RESULTS,
    model_name=MODEL,
    tsfm_context_length=256,           # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## RQ-M — representation-fairness ablation (`run_representation_fairness`)

Each model runs TWICE at full data, MSE, 3 seeds (CHANGES.md §35): **native**
(`channel_aggregation='concat'`, its own pooling — the campaign shape, so this arm is a
**cache hit**) vs **common** (`channel_aggregation='mean'`, `pooling='mean'` — one new
Stage-A embedding pass per dataset). Anchors: **FD001** (single-condition) + **FD004**
(multi-condition contrast) — the cross-TSFM ranking is checked on both regimes.
Restartable; per-session CSV so parallel sessions never share a file.


In [ ]:
from src.sweep import run_representation_fairness

for ds in ['FD001', 'FD004']:
    p = run_representation_fairness(
        config.replace(dataset=ds, sensor_columns=None),
        models=[MODEL], device=device,
        out_csv=f'{RESULTS}/representation_fairness_{TAG}.csv')
    print('fairness →', p)


In [ ]:
# Peek: this model's fairness rows (native vs common). The cross-TSFM RQ-M verdict is
# assembled later on a core runtime from ALL FIVE representation_fairness_*.csv files.
import pandas as pd
df = pd.read_csv(f'{RESULTS}/representation_fairness_{TAG}.csv')
print(df.groupby(['dataset', 'model', 'mode'])[['rmse_clipped', 'nasa_clipped']]
        .mean().round(2).to_string())
